<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, FileUpload, Output
from IPython.display import display, Markdown, HTML, clear_output
import io

# === 1. Oppsett ===
uploader = FileUpload(accept='', multiple=False, description="Last opp data")
app_display = Output()

def start_analysen(change):
    with app_display:
        clear_output(wait=True)
        if not uploader.value: return

        try:
            raw = uploader.value
            file_info = raw[0] if isinstance(raw, (list, tuple)) else list(raw.values())[0]
            content = file_info['content']
            df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')
            t_d, n_d = df.iloc[:,0].values, df.iloc[:,1].values
        except:
            print("Feil: Sjekk CSV-formatet."); return

        # === 2. ESTIMERING (10/85-regel) ===
        y0_v = n_d[0]
        A_v = n_d[-1] - y0_v
        t10 = t_d[np.where(n_d > y0_v + 0.10 * A_v)[0][0]]
        t85 = t_d[np.where(n_d > y0_v + 0.85 * A_v)[0][0]]
        t63 = t_d[np.where(n_d > y0_v + 0.63 * A_v)[0][0]]
        L_est = max(0, float(t10 - 0.05 * (t85 - t10)))
        T_est = max(0.1, float(t63 - L_est))

        # === 3. PLOTT-FUNKSJON ===
        plot_out = Output(layout={'width': '100%', 'max_width': '600px'})

        def update_plot(change=None):
            with plot_out:
                clear_output(wait=True)
                A, T, L, y0 = A_s.value, T_s.value, L_s.value, y0_s.value
                y_m = np.where(t_d < L, y0, y0 + A * (1 - np.exp(-(t_d - L) / T)))

                fig, ax = plt.subplots(figsize=(8, 5))
                ax.plot(t_d, n_d, "b.", markersize=3, alpha=0.3, label="Måledata")
                ax.plot(t_d, y_m, "r-", linewidth=2, label="Modell")

                # Linjer og verdier
                ax.axhline(y0, color='black', linestyle='--', alpha=0.4)
                ax.text(t_d[0], y0, f' y0={y0:.1f}', fontweight='bold', va='bottom')
                ax.axvline(L, color='orange', linestyle=':', linewidth=2)
                ax.text(L, y0, f' L={L:.1f}s', color='orange', fontweight='bold', ha='right')

                y63 = y0 + 0.63 * A
                ax.axhline(y63, color='green', linestyle=':', alpha=0.4)
                ax.axvline(L+T, color='green', linestyle=':', alpha=0.4)
                ax.text(L+T, y63, f' T={T:.1f}s', color='green', fontweight='bold', va='top')

                ax.vlines(t_d[-1], y0, y0+A, color='purple', linewidth=3)
                ax.text(t_d[-1], y0+A/2, f' Δy={A:.1f}', color='purple', fontweight='bold')

                ax.grid(True, alpha=0.2); ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå")
                plt.tight_layout(); plt.show()

        # === 4. WIDGETS ===
        style = {'description_width': '40px'}
        s_layout = {'width': '280px'}
        A_s = FloatSlider(value=A_v, min=min(0, A_v*0.2), max=max(A_v*2, 1), step=0.01, description="Δy", style=style, layout=s_layout)
        T_s = FloatSlider(value=T_est, min=0.1, max=max(T_est*4, 1), step=0.1, description="T", style=style, layout=s_layout)
        L_s = FloatSlider(value=L_est, min=0, max=t_d[-1]/2, step=0.1, description="L", style=style, layout=s_layout)
        y0_s = FloatSlider(value=y0_v, min=y0_v-10, max=y0_v+10, step=0.01, description="y0", style=style, layout=s_layout)

        for s in [A_s, T_s, L_s, y0_s]: s.observe(update_plot, "value")

        # === 5. SYMMETRISK TABELL OG LIGNINGER ===
        # Tabell med faste kolonnebredder (33%) for symmetri
        table_html = """
        <div style="font-family: sans-serif; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background: #fff; min-width: 320px;">
            <h4 style="text-align: center; margin-top: 0;">SIMC Regulering</h4>
            <table style="width: 100%; border-collapse: collapse; table-layout: fixed;">
                <tr style="background: #f4f4f4;">
                    <th style="width: 33%; padding: 8px; border: 1px solid #eee; text-align: center;">λ</th>
                    <th style="width: 33%; padding: 8px; border: 1px solid #eee; text-align: center;">Respons</th>
                    <th style="width: 33%; padding: 8px; border: 1px solid #eee; text-align: center;">Obs.</th>
                </tr>
                <tr><td style="padding: 8px; border: 1px solid #eee; text-align: center;">T / 2</td><td style="text-align: center; border: 1px solid #eee;">Rolig</td><td style="text-align: center; border: 1px solid #eee;">Robust</td></tr>
                <tr><td style="padding: 8px; border: 1px solid #eee; text-align: center;">T / 4</td><td style="text-align: center; border: 1px solid #eee;">Standard</td><td style="text-align: center; border: 1px solid #eee;">Balanse</td></tr>
                <tr><td style="padding: 8px; border: 1px solid #eee; text-align: center;">T / 6</td><td style="text-align: center; border: 1px solid #eee;">Rask</td><td style="text-align: center; border: 1px solid #eee;">Aggressiv</td></tr>
            </table>
        </div>
        """

        # Bruker Markdown-widget for ligningene for å få ekte LaTeX-utseende
        ligning_box = Output(layout={'margin': '10px 0 0 10px'})
        with ligning_box:
            display(Markdown(r"""
**PID Formler:**
- $K = \frac{\Delta y}{\Delta u}$
- $K_p = \frac{T}{K \cdot (\lambda + L)}$
- $T_i = \min(T, 4 \cdot (\lambda + L))$
            """))

        kontroller = VBox([A_s, T_s, L_s, y0_s], layout={'width': '300px'})
        info_kolonne = VBox([widgets.HTML(table_html), ligning_box], layout={'margin': '0 0 0 30px'})
        dashbord = HBox([plot_out, kontroller, info_kolonne], layout={'flex_flow': 'row wrap', 'align_items': 'flex-start'})

        display(dashbord)
        update_plot()

# === Start ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Simulator"), uploader, app_display)


# FOPDT Simulator

FileUpload(value={}, description='Last opp data')

Output()